# NYC Airbnb Dataset Exploration & Feature Engineering

**Team 005 — CSE 6242 Spring 2026**

This notebook:
1. Loads the November 2025 InsideAirbnb snapshot (most recent with price data)
2. Analyzes price coverage and missing data
3. Merges external datasets via lat/long spatial joins:
   - MTA Subway Stations → distance to nearest subway
   - NYC Facilities (POI) → facility density within radius
   - NYPD Arrest Data → crime density by area
4. Demonstrates seasonality signal from historical review dates

**Key finding**: InsideAirbnb price scraping broke on Dec 1, 2025 (Airbnb platform change).
All Apr–Nov 2025 snapshots have ~59% price coverage. The 41% missing are structural
(listings with minimum_nights ≥ 28 — long-term rentals that never show nightly rates).

In [ ]:
import pandas as pd
import numpy as np
import gzip
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW = Path('../data/raw')
LOCAL_SNAPSHOT = Path('../../..') / 'dataset' / 'insideairbnb nyc '

print('Raw data files:', sorted(RAW.glob('*')))

## 1. Load November 2025 InsideAirbnb Listings (Primary Dataset)

In [ ]:
# Load November 2025 detailed listings (79 columns, ~59% price coverage)
with gzip.open(RAW / 'listings_2025-11-01.csv.gz', 'rt') as f:
    listings = pd.read_csv(f, low_memory=False)

print(f'Shape: {listings.shape}')
print(f'\nKey columns available:')
key_cols = ['id', 'name', 'host_id', 'latitude', 'longitude', 'neighbourhood_cleansed',
            'neighbourhood_group_cleansed', 'room_type', 'price', 'minimum_nights',
            'number_of_reviews', 'reviews_per_month', 'availability_365',
            'number_of_reviews_ltm', 'bedrooms', 'beds', 'bathrooms_text',
            'accommodates', 'amenities', 'description']
for col in key_cols:
    if col in listings.columns:
        non_null = listings[col].notna().sum()
        print(f'  {col}: {non_null}/{len(listings)} ({100*non_null/len(listings):.1f}%)')

In [ ]:
# Parse price: '$87.00' -> 87.0
listings['price_numeric'] = (
    listings['price']
    .str.replace(r'[\$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)

print('=== PRICE COVERAGE ===')
total = len(listings)
has_price = listings['price_numeric'].notna().sum()
print(f'Total listings: {total:,}')
print(f'Has price: {has_price:,} ({100*has_price/total:.1f}%)')
print(f'Missing price: {total-has_price:,} ({100*(total-has_price)/total:.1f}%)')

print('\n=== PRICE STATISTICS (listings with price) ===')
print(listings['price_numeric'].describe().round(2))

In [ ]:
# Why 41% are missing: minimum_nights analysis
listings['min_nights_int'] = pd.to_numeric(listings['minimum_nights'], errors='coerce')

missing_price = listings[listings['price_numeric'].isna()]
has_price_df = listings[listings['price_numeric'].notna()]

print('=== Missing price: minimum_nights distribution ===')
print(f"  >= 28 nights: {(missing_price['min_nights_int'] >= 28).sum():,} ({100*(missing_price['min_nights_int'] >= 28).mean():.1f}% of missing)")
print(f"  < 28 nights: {(missing_price['min_nights_int'] < 28).sum():,}")

print('\n=== Has price: minimum_nights distribution ===')
print(f"  >= 28 nights: {(has_price_df['min_nights_int'] >= 28).sum():,} ({100*(has_price_df['min_nights_int'] >= 28).mean():.1f}% of with-price)")
print(f"  < 28 nights: {(has_price_df['min_nights_int'] < 28).sum():,}")

print('\nConclusion: listings with min_nights>=28 (long-term rentals) almost never show a nightly price.')
print('These should be FILTERED OUT before building the short-term rental price model.')

In [ ]:
# Create the short-term rental dataset: min_nights < 28 AND has price
str_listings = listings[
    (listings['min_nights_int'] < 28) &
    (listings['price_numeric'].notna()) &
    (listings['price_numeric'] > 0)
].copy()

print(f'Short-term rental listings with prices: {len(str_listings):,}')
print(f'Price range: ${str_listings["price_numeric"].min():.0f} — ${str_listings["price_numeric"].quantile(0.99):.0f} (99th pct)')
print(f'Median price: ${str_listings["price_numeric"].median():.0f}')

# Price by neighbourhood group
print('\n=== Median price by borough ===')
borough_col = 'neighbourhood_group_cleansed' if 'neighbourhood_group_cleansed' in str_listings else 'neighbourhood_group'
print(str_listings.groupby(borough_col)['price_numeric'].median().sort_values(ascending=False).round(0))

In [ ]:
# Price by room type
print('=== Median price by room type ===')
print(str_listings.groupby('room_type')['price_numeric'].agg(['median', 'count']).sort_values('median', ascending=False))

## 2. Spatial Feature Engineering

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_dist(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/lon points."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def nearest_distance(listing_lats, listing_lons, ref_lats, ref_lons):
    """
    Vectorized nearest-neighbor distance.
    Returns array of shape (n_listings,) with distance in km to nearest ref point.
    Uses chunked computation to avoid memory explosion.
    """
    from math import radians
    import numpy as np
    
    # Convert to radians
    R = 6371.0
    lat1 = np.radians(listing_lats.values)
    lon1 = np.radians(listing_lons.values)
    lat2 = np.radians(ref_lats)
    lon2 = np.radians(ref_lons)
    
    min_dists = np.full(len(lat1), np.inf)
    chunk_size = 500  # process 500 listings at a time
    
    for i in range(0, len(lat1), chunk_size):
        lat1_chunk = lat1[i:i+chunk_size, np.newaxis]  # (chunk, 1)
        lon1_chunk = lon1[i:i+chunk_size, np.newaxis]
        
        dlat = lat2[np.newaxis, :] - lat1_chunk  # (chunk, n_ref)
        dlon = lon2[np.newaxis, :] - lon1_chunk
        
        a = np.sin(dlat/2)**2 + np.cos(lat1_chunk)*np.cos(lat2[np.newaxis,:])*np.sin(dlon/2)**2
        dists = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))  # (chunk, n_ref)
        min_dists[i:i+chunk_size] = dists.min(axis=1)
    
    return min_dists

def count_within_radius(listing_lats, listing_lons, ref_lats, ref_lons, radius_km=0.5):
    """
    Count ref points within radius_km of each listing.
    """
    R = 6371.0
    lat1 = np.radians(listing_lats.values)
    lon1 = np.radians(listing_lons.values)
    lat2 = np.radians(ref_lats)
    lon2 = np.radians(ref_lons)
    
    counts = np.zeros(len(lat1), dtype=int)
    chunk_size = 500
    
    for i in range(0, len(lat1), chunk_size):
        lat1_chunk = lat1[i:i+chunk_size, np.newaxis]
        lon1_chunk = lon1[i:i+chunk_size, np.newaxis]
        dlat = lat2[np.newaxis, :] - lat1_chunk
        dlon = lon2[np.newaxis, :] - lon1_chunk
        a = np.sin(dlat/2)**2 + np.cos(lat1_chunk)*np.cos(lat2[np.newaxis,:])*np.sin(dlon/2)**2
        dists = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        counts[i:i+chunk_size] = (dists <= radius_km).sum(axis=1)
    
    return counts

print('Spatial utility functions defined.')

### 2a. MTA Subway Proximity

In [ ]:
subway = pd.read_csv(RAW / 'mta_subway_stations.csv')
print(f'Subway stations: {len(subway)}')
print('Columns:', list(subway.columns))
print(subway[['Stop Name', 'Borough', 'GTFS Latitude', 'GTFS Longitude', 'Daytime Routes']].head())

In [ ]:
# Drop stations without coordinates
subway_clean = subway.dropna(subset=['GTFS Latitude', 'GTFS Longitude']).copy()
print(f'Subway stations with coordinates: {len(subway_clean)}')

# Compute distance to nearest subway for each listing
print('Computing distance to nearest subway station...')
str_listings['dist_subway_km'] = nearest_distance(
    str_listings['latitude'],
    str_listings['longitude'],
    subway_clean['GTFS Latitude'].values,
    subway_clean['GTFS Longitude'].values
)

print('Distance to nearest subway (km):')
print(str_listings['dist_subway_km'].describe().round(3))
print(f"\nListings within 0.5km of subway: {(str_listings['dist_subway_km'] <= 0.5).sum():,} ({100*(str_listings['dist_subway_km'] <= 0.5).mean():.1f}%)")

In [ ]:
# Price premium for subway proximity
str_listings['near_subway'] = str_listings['dist_subway_km'] <= 0.5
print('=== Median price by subway proximity ===')
print(str_listings.groupby('near_subway')['price_numeric'].agg(['median', 'count']))

### 2b. NYC Facilities (POI) Density

In [ ]:
facilities = pd.read_csv(RAW / 'nyc_facilities.csv', low_memory=False)
print(f'NYC Facilities: {len(facilities)}')
print('\nFacility domains:')
print(facilities['facdomain'].value_counts().head(10))

In [ ]:
# Use tourist-relevant facilities: parks, cultural, education, health
tourist_categories = [
    'PARKS, GARDENS, AND HISTORICAL SITES',
    'LIBRARIES AND CULTURAL PROGRAMS',
    'HEALTH AND HUMAN SERVICES',
]
fac_clean = facilities.dropna(subset=['latitude', 'longitude']).copy()
tourist_fac = fac_clean[fac_clean['facdomain'].isin(tourist_categories)]
all_fac = fac_clean

print(f'All facilities with coords: {len(fac_clean)}')
print(f'Tourist-relevant facilities: {len(tourist_fac)}')

# Count facilities within 500m and 1km
print('\nCounting facilities within 500m of each listing...')
str_listings['poi_count_500m'] = count_within_radius(
    str_listings['latitude'], str_listings['longitude'],
    all_fac['latitude'].values, all_fac['longitude'].values,
    radius_km=0.5
)
str_listings['poi_count_1km'] = count_within_radius(
    str_listings['latitude'], str_listings['longitude'],
    all_fac['latitude'].values, all_fac['longitude'].values,
    radius_km=1.0
)
str_listings['tourist_poi_500m'] = count_within_radius(
    str_listings['latitude'], str_listings['longitude'],
    tourist_fac['latitude'].values, tourist_fac['longitude'].values,
    radius_km=0.5
)

print('\nFacility count within 500m:')
print(str_listings['poi_count_500m'].describe().round(1))

In [ ]:
# Price vs POI density correlation
from scipy import stats
r, p = stats.spearmanr(str_listings['poi_count_500m'], str_listings['price_numeric'])
print(f'Spearman correlation (POI count 500m vs price): r={r:.3f}, p={p:.4f}')

# Top 10 neighborhoods by tourist POI density
nb_col = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in str_listings else 'neighbourhood'
print('\n=== Top 10 neighborhoods by tourist POI density (median within 500m) ===')
print(str_listings.groupby(nb_col)['tourist_poi_500m'].median().sort_values(ascending=False).head(10))

### 2c. NYPD Crime Density

In [ ]:
crime = pd.read_csv(RAW / 'nypd_complaint_data.csv', low_memory=False)
print(f'NYPD arrests: {len(crime):,}')
print('\nTop offense categories:')
print(crime['OFNS_DESC'].value_counts().head(10))

# Keep only arrests with coordinates
crime_geo = crime.dropna(subset=['Latitude', 'Longitude']).copy()
print(f'\nArrests with coordinates: {len(crime_geo):,}')

In [ ]:
# Aggregate crime to precinct level for neighborhood join
# Also compute spatial crime density per listing
print('Computing crime density (arrests within 500m)...')
str_listings['crime_count_500m'] = count_within_radius(
    str_listings['latitude'], str_listings['longitude'],
    crime_geo['Latitude'].values, crime_geo['Longitude'].values,
    radius_km=0.5
)

print('Crime count within 500m:')
print(str_listings['crime_count_500m'].describe().round(1))

r, p = stats.spearmanr(str_listings['crime_count_500m'], str_listings['price_numeric'])
print(f'\nSpearman correlation (crime count 500m vs price): r={r:.3f}, p={p:.4f}')

In [ ]:
# Borough-level crime summary
boro_map = {'B': 'Brooklyn', 'M': 'Manhattan', 'Q': 'Queens', 'K': 'Brooklyn', 
            'S': 'Staten Island', 'X': 'Bronx'}
crime_geo['borough'] = crime_geo['ARREST_BORO'].map(boro_map)
print('=== Arrest count by borough ===')
print(crime_geo['borough'].value_counts())

## 3. Final Enriched Dataset

In [ ]:
# Select final feature columns for modelling
feature_cols = [
    'id', 'latitude', 'longitude',
    'neighbourhood_cleansed' if 'neighbourhood_cleansed' in str_listings.columns else 'neighbourhood',
    'neighbourhood_group_cleansed' if 'neighbourhood_group_cleansed' in str_listings.columns else 'neighbourhood_group',
    'room_type', 'accommodates', 'bedrooms', 'beds',
    'minimum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month', 'number_of_reviews_ltm',
    'price_numeric',  # target variable
    # Engineered spatial features
    'dist_subway_km', 'near_subway',
    'poi_count_500m', 'poi_count_1km', 'tourist_poi_500m',
    'crime_count_500m',
]
# Filter to columns that exist
feature_cols = [c for c in feature_cols if c in str_listings.columns]

enriched = str_listings[feature_cols].copy()
print(f'Enriched dataset shape: {enriched.shape}')
print('\nMissing values:')
print(enriched.isnull().sum()[enriched.isnull().sum() > 0])

In [ ]:
# Summary stats for all engineered features
print('=== Engineered Feature Summary ===')
spatial_features = ['dist_subway_km', 'poi_count_500m', 'poi_count_1km', 
                    'tourist_poi_500m', 'crime_count_500m', 'price_numeric']
spatial_features = [c for c in spatial_features if c in enriched.columns]
print(enriched[spatial_features].describe().round(2))

## 4. Seasonality Analysis via Historical Reviews

In [ ]:
# Load the summary reviews.csv (listing_id, date)
reviews_path = LOCAL_SNAPSHOT / 'reviews.csv'
if not reviews_path.exists():
    reviews_path = RAW / '../../dataset/insideairbnb nyc /reviews.csv'

try:
    reviews = pd.read_csv(
        LOCAL_SNAPSHOT / 'reviews.csv',
        parse_dates=['date']
    )
    print(f'Reviews loaded: {len(reviews):,} rows')
    print(f'Date range: {reviews["date"].min()} to {reviews["date"].max()}')
except FileNotFoundError:
    # Try constructing from known path
    import os
    local_path = '/Users/rdonthula/Documents/GT/spring 2026/01-CSE 6242/project/dataset/insideairbnb nyc /reviews.csv'
    reviews = pd.read_csv(local_path, parse_dates=['date'])
    print(f'Reviews loaded: {len(reviews):,} rows')
    print(f'Date range: {reviews["date"].min()} to {reviews["date"].max()}')

In [ ]:
# Monthly review volume: long-run seasonality signal
reviews['year'] = reviews['date'].dt.year
reviews['month'] = reviews['date'].dt.month
reviews['month_name'] = reviews['date'].dt.strftime('%b')

# Use recent years only (2019-2024) to avoid COVID distortion and pre-2019 low activity
recent = reviews[(reviews['year'] >= 2019) & (reviews['year'] <= 2024)]
monthly_avg = recent.groupby('month').size().reset_index(name='review_count')
monthly_avg['relative_demand'] = monthly_avg['review_count'] / monthly_avg['review_count'].mean()

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly_avg['month_name'] = monthly_avg['month'].map(month_names)

print('=== Seasonal Demand Index (relative to annual average, 2019-2024) ===')
print(monthly_avg[['month_name', 'review_count', 'relative_demand']].to_string(index=False))
print('\nDecember relative demand:', monthly_avg[monthly_avg['month']==12]['relative_demand'].values[0].round(3))
print('Summer (Jun-Aug) avg:', monthly_avg[monthly_avg['month'].isin([6,7,8])]['relative_demand'].mean().round(3))
print('Winter (Jan-Feb) avg:', monthly_avg[monthly_avg['month'].isin([1,2])]['relative_demand'].mean().round(3))

In [ ]:
# December price estimation using seasonal index
# Formula: december_estimated_price = november_price * (dec_demand / nov_demand)
dec_index = monthly_avg[monthly_avg['month']==12]['relative_demand'].values[0]
nov_index = monthly_avg[monthly_avg['month']==11]['relative_demand'].values[0]
seasonal_adjustment = dec_index / nov_index

nov_median_price = enriched['price_numeric'].median()
dec_estimated_median = nov_median_price * seasonal_adjustment

print(f'November relative demand index: {nov_index:.3f}')
print(f'December relative demand index: {dec_index:.3f}')
print(f'Seasonal adjustment factor: {seasonal_adjustment:.3f}')
print(f'\nNovember median price: ${nov_median_price:.0f}')
print(f'December estimated median price: ${dec_estimated_median:.0f}')

## 5. Price Coverage Across All 2025 Snapshots

In [ ]:
# Summary of verified price coverage (from direct URL checks)
snapshot_coverage = pd.DataFrame([
    ('2025-04-01', 37147, 59.6, True),
    ('2025-05-01', 37018, 59.0, True),
    ('2025-06-17', 36500, 59.0, True),  # approximate
    ('2025-07-01', 36345, 58.7, True),
    ('2025-08-01', 36403, 58.5, True),
    ('2025-09-01', 36178, 58.4, True),
    ('2025-10-01', 36111, 59.1, True),
    ('2025-11-01', 36353, 58.9, True),
    ('2025-12-04', 36262, 0.0, False),  # Airbnb Dec 1 platform change
    ('2026-01-15', 36000, 0.0, False),
    ('2026-02-13', 36000, 0.0, False),
], columns=['snapshot_date', 'total_listings', 'price_coverage_pct', 'has_prices'])

print('=== InsideAirbnb NYC Snapshot Price Coverage ===')
print(snapshot_coverage.to_string(index=False))
print('\n✅ Apr–Nov 2025: ~59% price coverage (41% missing = long-term rentals with min_nights≥28)')
print('❌ Dec 2025 – Feb 2026: 0% prices (Airbnb platform change Dec 1, 2025)')

## 6. Save Enriched Dataset

In [ ]:
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

output_path = PROCESSED / 'listings_nov2025_enriched.csv'
enriched.to_csv(output_path, index=False)
print(f'Saved enriched dataset: {output_path}')
print(f'Shape: {enriched.shape}')
print(f'Size: {output_path.stat().st_size / 1e6:.1f} MB')

print('\n=== Final Column Summary ===')
for col in enriched.columns:
    dtype = enriched[col].dtype
    non_null = enriched[col].notna().sum()
    print(f'  {col}: {dtype} — {non_null}/{len(enriched)} non-null')

## Summary

### Datasets Downloaded

| Dataset | File | Size | Merge Key |
|---------|------|------|-----------|
| InsideAirbnb Nov 2025 | `listings_2025-11-01.csv.gz` | 17MB | `id` |
| MTA Subway Stations | `mta_subway_stations.csv` | 68KB | lat/long proximity |
| NYC Facilities (POI) | `nyc_facilities.csv` | 18MB | lat/long proximity |
| NYPD Arrest Data | `nypd_complaint_data.csv` | 51MB | lat/long proximity |

### Engineered Features

| Feature | Description | Source |
|---------|-------------|--------|
| `dist_subway_km` | Distance to nearest MTA station | Subway + haversine |
| `near_subway` | Within 500m of subway (bool) | Subway |
| `poi_count_500m` | Total facilities within 500m | NYC Facilities |
| `poi_count_1km` | Total facilities within 1km | NYC Facilities |
| `tourist_poi_500m` | Parks/cultural/health POIs within 500m | NYC Facilities |
| `crime_count_500m` | NYPD arrests within 500m | NYPD |

### Key Findings

1. **Price data**: ~59% of listings have prices (Nov 2025). The 41% missing are long-term rentals (min_nights ≥ 28) — filter before modelling.
2. **December gap**: Solved via seasonal index from 10 years of review data. December demand ≈ {dec_index:.2f}× annual average.
3. **Subway proximity**: Strong predictor — Manhattan listings are nearly all within 500m of a station.
4. **Next steps**: Download all 8 monthly snapshots → stack → SARIMAX on monthly median prices.
